# Practice (DuckDB): the same batch pipeline in SQL

The pandas practice notebook (`practice_pandas.ipynb`) rebuilt in SQL. The outputs should match row for row, so you can compare the two engines.

**Every table goes through the same six ingestion steps.** These are the "What to brush up on" points from the candidate prep pack, one step each:

| step | prep-pack point | creates |
|---|---|---|
| x.1 | Reading a CSV and taking control of the column types | `<table>_raw` |
| x.2 | Tidying strings: whitespace and casing | `<table>_tidy` |
| x.3 | Missing data | `<table>_nulls` |
| x.4 | Converting text to dates and numbers when some values won't convert | `<table>_typed` |
| x.5 | Removing duplicate rows | `<table>_dedup` |
| x.6 | Filtering with boolean rules: valid vs quarantine | `<table>_clean` + `<table>_quarantine` |

After that come the two cross-table points: **3. join** and **4. group, aggregate, sort**. Then **5. run end to end**, **6. tests** and **notes** on the four discussion questions.

## 0. Install and configure

DuckDB is a dependency of this repo (`pyproject.toml`), so `uv sync` already installed it into the kernel's environment.


Best practice from the DuckDB Python docs: use an **explicit connection** instead of the module-level default. Set the settings that affect results **explicitly**, so a run doesn't depend on the machine it runs on, and print them so the run records its own configuration.

The database is in memory: the job's outputs are CSV files, and every run rebuilds from the raw files.

In [1]:
import json
from pathlib import Path

import duckdb

# Repo root = the nearest folder with data/input, so this runs from the repo root or from bi/jupyter/.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "input").is_dir())
RAW = ROOT / "data" / "input"
OUT_DIR = ROOT / "data" / "output" / "practice_duckdb"

CONFIG = {
    "threads": 4,                       # pinned: same plan on any machine
    "memory_limit": "1GB",              # DuckDB spills to disk beyond this instead of crashing
    "preserve_insertion_order": True,   # the default; source_row below relies on it
    "TimeZone": "UTC",                  # timestamps don't depend on the laptop's locale
}

con = duckdb.connect(config=CONFIG)
print(duckdb.__version__)
con.sql("SELECT name, value FROM duckdb_settings() WHERE name IN ('threads', 'memory_limit', 'preserve_insertion_order', 'TimeZone')")

2.0.1


nameVARCHAR,valueVARCHAR
memory_limit,953.6 MiB
preserve_insertion_order,true
threads,4
TimeZone,UTC


One small helper keeps every step the same shape: it creates (or replaces) a view or table and shows the result. It also records the SQL, so `run()` in section 5 can replay the whole pipeline on a fresh connection.

In [2]:
STEPS: dict[str, str] = {}  # name -> SQL, in the order the steps were defined


def step(name: str, select: str, kind: str = "VIEW") -> duckdb.DuckDBPyRelation:
    """CREATE OR REPLACE one pipeline step, remember it for run(), and show it."""
    sql = f"CREATE OR REPLACE {kind} {name} AS\n{select}"
    con.execute(sql)
    STEPS[name] = sql
    return con.table(name)

## 1. Customers

### 1.1 Read with controlled types

- `all_varchar = true`: nothing is inferred, so `"00042"` stays `"00042"` and dates stay text until step 1.4.
- `header`, `delim` and `quote` are set explicitly. The docs say explicit options bypass the sniffer for that setting, so after looking at the file we pin what we saw.
- DuckDB only treats an empty field as NULL, so Namibia's `"NA"` survives. pandas needs `keep_default_na=False` for the same thing.
- `source_row` is the line number in the file, for lineage. `row_number() OVER ()` follows file order because `preserve_insertion_order` is on.
- This step is a **table**, so the file is read once.

In [3]:
step("customers_raw", f"""
SELECT row_number() OVER () + 1 AS source_row, *
FROM read_csv('{RAW / "customers.csv"}', header = true, delim = ',', quote = '"', all_varchar = true)
""", kind="TABLE")

source_rowBIGINT,customer_idVARCHAR,nameVARCHAR,emailVARCHAR,countryVARCHAR,signup_dateVARCHAR,updated_atVARCHAR
2,00017,Alice Smith,ALICE@example.com,GB,2023-01-15,2024-01-01T09:00:00
3,00042,Bob Jones,bob@example.com,gb,2023-02-20,2024-01-01T09:00:00
4,00042,Bob Jones,robert.jones@example.com,GB,2023-02-20,2024-03-10T12:30:00
5,00073,José Núñez,jose@example.es,ES,2023-03-05,2024-01-01T09:00:00
6,00088,Nangula Shikongo,nangula@example.na,NA,2023-04-11,2024-01-01T09:00:00
7,00091,Chen Wei,chen@example.cn,cn,05/06/2023,2024-01-01T09:00:00
8,00091,Chen Wei,chen@example.cn,cn,05/06/2023,2024-01-01T09:00:00
9,00105,NULL,anon@example.com,NULL,2023-07-01,2024-01-01T09:00:00


In [4]:
con.sql("SUMMARIZE customers_raw")  # profile: distinct values, nulls, min/max per column

column_nameVARCHAR,column_typeVARCHAR,minVARCHAR,maxVARCHAR,approx_uniqueBIGINT,avgVARCHAR,stdVARCHAR,q25VARCHAR,q50VARCHAR,q75VARCHAR,countBIGINT,"null_percentageDECIMAL(9,2)"
source_row,BIGINT,2,9,8,5.5,2.449489742783178,4,6,8,8,0.00
customer_id,VARCHAR,00017,00105,6,NULL,NULL,NULL,NULL,NULL,8,0.00
name,VARCHAR,Alice Smith,Nangula Shikongo,5,NULL,NULL,NULL,NULL,NULL,8,12.50
email,VARCHAR,ALICE@example.com,robert.jones@example.com,7,NULL,NULL,NULL,NULL,NULL,8,0.00
country,VARCHAR,cn,gb,5,NULL,NULL,NULL,NULL,NULL,8,12.50
signup_date,VARCHAR,05/06/2023,2023-07-01,6,NULL,NULL,NULL,NULL,NULL,8,0.00
updated_at,VARCHAR,2024-01-01T09:00:00,2024-03-10T12:30:00,2,NULL,NULL,NULL,NULL,NULL,8,0.00


### 1.2 Tidy strings
Trim every text column. Emails go lowercase, countries uppercase (`" cn"` becomes `"CN"`, `"gb "` becomes `"GB"`).

In [5]:
step("customers_tidy", """
SELECT source_row,
       trim(customer_id)    AS customer_id,
       trim(name)           AS name,
       lower(trim(email))   AS email,
       upper(trim(country)) AS country,
       trim(signup_date)    AS signup_date,
       trim(updated_at)     AS updated_at
FROM customers_raw
""")

source_rowBIGINT,customer_idVARCHAR,nameVARCHAR,emailVARCHAR,countryVARCHAR,signup_dateVARCHAR,updated_atVARCHAR
2,00017,Alice Smith,alice@example.com,GB,2023-01-15,2024-01-01T09:00:00
3,00042,Bob Jones,bob@example.com,GB,2023-02-20,2024-01-01T09:00:00
4,00042,Bob Jones,robert.jones@example.com,GB,2023-02-20,2024-03-10T12:30:00
5,00073,José Núñez,jose@example.es,ES,2023-03-05,2024-01-01T09:00:00
6,00088,Nangula Shikongo,nangula@example.na,NA,2023-04-11,2024-01-01T09:00:00
7,00091,Chen Wei,chen@example.cn,CN,05/06/2023,2024-01-01T09:00:00
8,00091,Chen Wei,chen@example.cn,CN,05/06/2023,2024-01-01T09:00:00
9,00105,NULL,anon@example.com,NULL,2023-07-01,2024-01-01T09:00:00


### 1.3 Missing data
After trimming, a field that held only spaces is `''`, so `''` becomes NULL as well. This SQL is identical for every table: `COLUMNS(...)` applies `nullif` to each column and keeps its name.

In [6]:
step("customers_nulls", """
SELECT source_row, nullif(COLUMNS(* EXCLUDE (source_row)), '')
FROM customers_tidy
""")

source_rowBIGINT,customer_idVARCHAR,nameVARCHAR,emailVARCHAR,countryVARCHAR,signup_dateVARCHAR,updated_atVARCHAR
2,00017,Alice Smith,alice@example.com,GB,2023-01-15,2024-01-01T09:00:00
3,00042,Bob Jones,bob@example.com,GB,2023-02-20,2024-01-01T09:00:00
4,00042,Bob Jones,robert.jones@example.com,GB,2023-02-20,2024-03-10T12:30:00
5,00073,José Núñez,jose@example.es,ES,2023-03-05,2024-01-01T09:00:00
6,00088,Nangula Shikongo,nangula@example.na,NA,2023-04-11,2024-01-01T09:00:00
7,00091,Chen Wei,chen@example.cn,CN,05/06/2023,2024-01-01T09:00:00
8,00091,Chen Wei,chen@example.cn,CN,05/06/2023,2024-01-01T09:00:00
9,00105,NULL,anon@example.com,NULL,2023-07-01,2024-01-01T09:00:00


### 1.4 Convert text to dates
`try_strptime` returns NULL instead of raising when a value doesn't match the format. Only ISO dates are accepted, so nothing is guessed. The text columns stay next to the typed ones so you can see what failed.

In [7]:
step("customers_typed", """
SELECT *,
       try_strptime(signup_date, '%Y-%m-%d')::DATE       AS signup_date_typed,
       try_strptime(updated_at, '%Y-%m-%dT%H:%M:%S')     AS updated_at_typed
FROM customers_nulls
""")
con.sql("SELECT source_row, customer_id, signup_date FROM customers_typed WHERE signup_date IS NOT NULL AND signup_date_typed IS NULL")

source_rowBIGINT,customer_idVARCHAR,signup_dateVARCHAR
7,00091,05/06/2023
8,00091,05/06/2023


### 1.5 Remove duplicates
One row per `customer_id`: the latest `updated_at` wins, and the file line breaks ties. `QUALIFY` filters on a window function without a subquery.

In [8]:
con.sql("SELECT source_row, customer_id, email, updated_at FROM customers_typed QUALIFY count(*) OVER (PARTITION BY customer_id) > 1")

source_rowBIGINT,customer_idVARCHAR,emailVARCHAR,updated_atVARCHAR
3,00042,bob@example.com,2024-01-01T09:00:00
4,00042,robert.jones@example.com,2024-03-10T12:30:00
7,00091,chen@example.cn,2024-01-01T09:00:00
8,00091,chen@example.cn,2024-01-01T09:00:00


In [9]:
step("customers_dedup", """
SELECT *
FROM customers_typed
QUALIFY row_number() OVER (
    PARTITION BY customer_id ORDER BY updated_at_typed DESC NULLS LAST, source_row DESC
) = 1
""")

source_rowBIGINT,customer_idVARCHAR,nameVARCHAR,emailVARCHAR,countryVARCHAR,signup_dateVARCHAR,updated_atVARCHAR,signup_date_typedDATE,updated_at_typedTIMESTAMP
9,00105,NULL,anon@example.com,NULL,2023-07-01,2024-01-01T09:00:00,2023-07-01,2024-01-01 09:00:00
2,00017,Alice Smith,alice@example.com,GB,2023-01-15,2024-01-01T09:00:00,2023-01-15,2024-01-01 09:00:00
4,00042,Bob Jones,robert.jones@example.com,GB,2023-02-20,2024-03-10T12:30:00,2023-02-20,2024-03-10 12:30:00
8,00091,Chen Wei,chen@example.cn,CN,05/06/2023,2024-01-01T09:00:00,NULL,2024-01-01 09:00:00
5,00073,José Núñez,jose@example.es,ES,2023-03-05,2024-01-01T09:00:00,2023-03-05,2024-01-01 09:00:00
6,00088,Nangula Shikongo,nangula@example.na,NA,2023-04-11,2024-01-01T09:00:00,2023-04-11,2024-01-01 09:00:00


### 1.6 Filter: valid vs quarantine
Each rule is a `CASE` that names itself. `concat_ws` joins the names that fired and skips the NULLs, so `''` means the row passed every rule.

Only a missing key quarantines a customer. A bad `signup_date` is kept as NULL, because the field isn't needed downstream and dropping the customer would turn their orders into "unknown customer".

In [10]:
step("customers_checked", """
SELECT *,
       concat_ws(';',
           CASE WHEN customer_id IS NULL THEN 'missing_customer_id' END
       ) AS reject_reason
FROM customers_dedup
""")
step("customers_quarantine", """
SELECT source_row, customer_id, name, email, country, signup_date, updated_at, reject_reason
FROM customers_checked
WHERE reject_reason <> ''
ORDER BY source_row
""", kind="TABLE")
step("customers_clean", """
SELECT customer_id, name, email, country,
       signup_date_typed AS signup_date,
       updated_at_typed  AS updated_at
FROM customers_checked
WHERE reject_reason = ''
ORDER BY customer_id
""", kind="TABLE")

customer_idVARCHAR,nameVARCHAR,emailVARCHAR,countryVARCHAR,signup_dateDATE,updated_atTIMESTAMP
00017,Alice Smith,alice@example.com,GB,2023-01-15,2024-01-01 09:00:00
00042,Bob Jones,robert.jones@example.com,GB,2023-02-20,2024-03-10 12:30:00
00073,José Núñez,jose@example.es,ES,2023-03-05,2024-01-01 09:00:00
00088,Nangula Shikongo,nangula@example.na,NA,2023-04-11,2024-01-01 09:00:00
00091,Chen Wei,chen@example.cn,CN,NULL,2024-01-01 09:00:00
00105,NULL,anon@example.com,NULL,2023-07-01,2024-01-01 09:00:00


## 2. Orders
The same six steps.

### 2.1 Read with controlled types

In [11]:
step("orders_raw", f"""
SELECT row_number() OVER () + 1 AS source_row, *
FROM read_csv('{RAW / "orders.csv"}', header = true, delim = ',', quote = '"', all_varchar = true)
""", kind="TABLE")

source_rowBIGINT,order_idVARCHAR,customer_idVARCHAR,order_dateVARCHAR,amountVARCHAR,quantityVARCHAR,statusVARCHAR,notesVARCHAR
2,1001,00017,2024-03-01,12.50,1,Shipped,NULL
3,1002,00042,2024-03-01,"£1,200.00",2,shipped,"gift wrap, urgent"
4,1003,00073,2024-03-02,30,3,DELIVERED,NULL
5,1004,00088,2024-03-02,45.00,1,delivered,NULL
6,1005,00091,2024-02-30,20.00,1,shipped,NULL
7,1006,00017,2024-03-03,N/A,1,shipped,NULL
8,1007,00999,2024-03-03,15.00,1,shipped,customer not in master
9,1008,00042,2024-03-04,-5.00,1,refunded,NULL
10,1009,00073,NULL,8.00,1,shipped,NULL
11,1010,00088,2024-03-04,9.99,NULL,shipped,NULL


In [12]:
con.sql("SUMMARIZE orders_raw")

column_nameVARCHAR,column_typeVARCHAR,minVARCHAR,maxVARCHAR,approx_uniqueBIGINT,avgVARCHAR,stdVARCHAR,q25VARCHAR,q50VARCHAR,q75VARCHAR,countBIGINT,"null_percentageDECIMAL(9,2)"
source_row,BIGINT,2,21,20,11.5,5.916079783099616,6,12,16,20,0.00
order_id,VARCHAR,1001,1018,18,NULL,NULL,NULL,NULL,NULL,20,0.00
customer_id,VARCHAR,00017,17,8,NULL,NULL,NULL,NULL,NULL,20,0.00
order_date,VARCHAR,06/03/2024,2024-03-06,8,NULL,NULL,NULL,NULL,NULL,20,5.00
amount,VARCHAR,-5.00,"£1,200.00",19,NULL,NULL,NULL,NULL,NULL,20,0.00
quantity,VARCHAR,1,two,4,NULL,NULL,NULL,NULL,NULL,20,5.00
status,VARCHAR,Cancelled,shipped,9,NULL,NULL,NULL,NULL,NULL,20,0.00
notes,VARCHAR,amended?,"gift wrap, urgent",3,NULL,NULL,NULL,NULL,NULL,20,85.00


### 2.2 Tidy strings
Trim everything. `status` goes lowercase, because `Shipped`, `shipped ` and `DELIVERED` are spellings of the same values.

In [13]:
step("orders_tidy", """
SELECT source_row,
       trim(order_id)     AS order_id,
       trim(customer_id)  AS customer_id,
       trim(order_date)   AS order_date,
       trim(amount)       AS amount,
       trim(quantity)     AS quantity,
       lower(trim(status)) AS status,
       trim(notes)        AS notes
FROM orders_raw
""")

source_rowBIGINT,order_idVARCHAR,customer_idVARCHAR,order_dateVARCHAR,amountVARCHAR,quantityVARCHAR,statusVARCHAR,notesVARCHAR
2,1001,00017,2024-03-01,12.50,1,shipped,NULL
3,1002,00042,2024-03-01,"£1,200.00",2,shipped,"gift wrap, urgent"
4,1003,00073,2024-03-02,30,3,delivered,NULL
5,1004,00088,2024-03-02,45.00,1,delivered,NULL
6,1005,00091,2024-02-30,20.00,1,shipped,NULL
7,1006,00017,2024-03-03,N/A,1,shipped,NULL
8,1007,00999,2024-03-03,15.00,1,shipped,customer not in master
9,1008,00042,2024-03-04,-5.00,1,refunded,NULL
10,1009,00073,NULL,8.00,1,shipped,NULL
11,1010,00088,2024-03-04,9.99,NULL,shipped,NULL


### 2.3 Missing data
The same SQL as 1.3. `"N/A"` in `amount` is *not* treated as missing here: it's a value that won't convert, and step 2.4 flags it as such.

In [14]:
step("orders_nulls", """
SELECT source_row, nullif(COLUMNS(* EXCLUDE (source_row)), '')
FROM orders_tidy
""")

source_rowBIGINT,order_idVARCHAR,customer_idVARCHAR,order_dateVARCHAR,amountVARCHAR,quantityVARCHAR,statusVARCHAR,notesVARCHAR
2,1001,00017,2024-03-01,12.50,1,shipped,NULL
3,1002,00042,2024-03-01,"£1,200.00",2,shipped,"gift wrap, urgent"
4,1003,00073,2024-03-02,30,3,delivered,NULL
5,1004,00088,2024-03-02,45.00,1,delivered,NULL
6,1005,00091,2024-02-30,20.00,1,shipped,NULL
7,1006,00017,2024-03-03,N/A,1,shipped,NULL
8,1007,00999,2024-03-03,15.00,1,shipped,customer not in master
9,1008,00042,2024-03-04,-5.00,1,refunded,NULL
10,1009,00073,NULL,8.00,1,shipped,NULL
11,1010,00088,2024-03-04,9.99,NULL,shipped,NULL


### 2.4 Convert text to dates and numbers
- Dates: `try_strptime` with one strict format. `2024-02-30` and the ambiguous `06/03/2024` both become NULL.
- Amounts: strip the currency symbols and thousands separators, then `TRY_CAST` to `DECIMAL(12, 2)`. Money is stored as an exact decimal, not a float.
- Quantity: `TRY_CAST('1.5' AS INTEGER)` silently **rounds to 2**, so only whole-number text is cast.

In [15]:
step("orders_typed", """
SELECT *,
       try_strptime(order_date, '%Y-%m-%d')::DATE                          AS order_date_typed,
       TRY_CAST(regexp_replace(amount, '[£$€,]', '', 'g') AS DECIMAL(12, 2)) AS amount_typed,
       CASE WHEN regexp_full_match(quantity, '[0-9]+') THEN quantity::INTEGER END AS quantity_typed
FROM orders_nulls
""")
con.sql("""
SELECT source_row, order_id, order_date, amount, quantity
FROM orders_typed
WHERE (order_date IS NOT NULL AND order_date_typed IS NULL)
   OR (amount IS NOT NULL AND amount_typed IS NULL)
   OR (quantity IS NOT NULL AND quantity_typed IS NULL)
""")

source_rowBIGINT,order_idVARCHAR,order_dateVARCHAR,amountVARCHAR,quantityVARCHAR
6,1005,2024-02-30,20.00,1
7,1006,2024-03-03,N/A,1
12,1011,2024-03-05,10.00,two
19,1016,06/03/2024,11.00,1


### 2.5 Remove duplicates
Rows that are identical in every business column are noise: keep the first and count the rest. Rows that share an `order_id` but *differ* are a conflict, which step 2.6 deals with.

In [16]:
con.sql("SELECT source_row, order_id, amount, notes FROM orders_typed QUALIFY count(*) OVER (PARTITION BY order_id) > 1")

source_rowBIGINT,order_idVARCHAR,amountVARCHAR,notesVARCHAR
13,1012,7.25,NULL
14,1012,7.25,NULL
15,1013,100.00,NULL
16,1013,110.00,amended?


In [17]:
step("orders_dedup", """
SELECT *
FROM orders_typed
QUALIFY row_number() OVER (
    PARTITION BY order_id, customer_id, order_date, amount, quantity, status, notes
    ORDER BY source_row
) = 1
""")

source_rowBIGINT,order_idVARCHAR,customer_idVARCHAR,order_dateVARCHAR,amountVARCHAR,quantityVARCHAR,statusVARCHAR,notesVARCHAR,order_date_typedDATE,"amount_typedDECIMAL(12,2)",quantity_typedINTEGER
2,1001,00017,2024-03-01,12.50,1,shipped,NULL,2024-03-01,12.50,1
8,1007,00999,2024-03-03,15.00,1,shipped,customer not in master,2024-03-03,15.00,1
15,1013,00073,2024-03-05,100.00,1,shipped,NULL,2024-03-05,100.00,1
20,1017,00105,2024-03-06,4.00,1,shipped,NULL,2024-03-06,4.00,1
3,1002,00042,2024-03-01,"£1,200.00",2,shipped,"gift wrap, urgent",2024-03-01,1200.00,2
4,1003,00073,2024-03-02,30,3,delivered,NULL,2024-03-02,30.00,3
5,1004,00088,2024-03-02,45.00,1,delivered,NULL,2024-03-02,45.00,1
6,1005,00091,2024-02-30,20.00,1,shipped,NULL,NULL,20.00,1
9,1008,00042,2024-03-04,-5.00,1,refunded,NULL,2024-03-04,-5.00,1
12,1011,00017,2024-03-05,10.00,two,shipped,NULL,2024-03-05,10.00,NULL


### 2.6 Filter: valid vs quarantine
Two SQL traps to avoid here:
- `status NOT IN (...)` is NULL, not true, when `status` is NULL. A missing status would pass silently without the `IS NULL` check.
- `customer_id NOT IN (SELECT ...)` returns no rows if the subquery contains a single NULL. `NOT EXISTS` doesn't have that problem.

In [18]:
step("orders_checked", """
SELECT o.*,
       concat_ws(';',
           CASE WHEN count(*) OVER (PARTITION BY order_id) > 1             THEN 'conflicting_duplicate_order_id' END,
           CASE WHEN order_date IS NULL                                     THEN 'missing_order_date' END,
           CASE WHEN order_date IS NOT NULL AND order_date_typed IS NULL    THEN 'unparseable_order_date' END,
           CASE WHEN amount_typed IS NULL                                   THEN 'unparseable_amount' END,
           CASE WHEN amount_typed < 0 AND status IS DISTINCT FROM 'refunded' THEN 'negative_amount' END,
           CASE WHEN quantity IS NULL                                       THEN 'missing_quantity' END,
           CASE WHEN quantity IS NOT NULL AND quantity_typed IS NULL        THEN 'unparseable_quantity' END,
           CASE WHEN status IS NULL
                  OR status NOT IN ('shipped', 'delivered', 'refunded', 'cancelled') THEN 'invalid_status' END,
           CASE WHEN NOT EXISTS (SELECT 1 FROM customers_clean c WHERE c.customer_id = o.customer_id)
                                                                            THEN 'unknown_customer' END
       ) AS reject_reason
FROM orders_dedup o
""")
step("orders_quarantine", """
SELECT source_row, order_id, customer_id, order_date, amount, quantity, status, notes, reject_reason
FROM orders_checked
WHERE reject_reason <> ''
ORDER BY source_row
""", kind="TABLE")
step("orders_clean", """
SELECT order_id, customer_id,
       order_date_typed AS order_date,
       amount_typed     AS amount,
       quantity_typed   AS quantity,
       status, notes, source_row
FROM orders_checked
WHERE reject_reason = ''
ORDER BY order_id
""", kind="TABLE")

order_idVARCHAR,customer_idVARCHAR,order_dateDATE,"amountDECIMAL(12,2)",quantityINTEGER,statusVARCHAR,notesVARCHAR,source_rowBIGINT
1001,00017,2024-03-01,12.50,1,shipped,NULL,2
1002,00042,2024-03-01,1200.00,2,shipped,"gift wrap, urgent",3
1003,00073,2024-03-02,30.00,3,delivered,NULL,4
1004,00088,2024-03-02,45.00,1,delivered,NULL,5
1008,00042,2024-03-04,-5.00,1,refunded,NULL,9
1012,00042,2024-03-05,7.25,1,shipped,NULL,13
1017,00105,2024-03-06,4.00,1,shipped,NULL,20
1018,00073,2024-03-06,50.00,1,cancelled,NULL,21


In [19]:
con.table("orders_quarantine")  # the raw text is kept, so anyone can see exactly what arrived

source_rowBIGINT,order_idVARCHAR,customer_idVARCHAR,order_dateVARCHAR,amountVARCHAR,quantityVARCHAR,statusVARCHAR,notesVARCHAR,reject_reasonVARCHAR
6,1005,00091,2024-02-30,20.00,1,shipped,NULL,unparseable_order_date
7,1006,00017,2024-03-03,N/A,1,shipped,NULL,unparseable_amount
8,1007,00999,2024-03-03,15.00,1,shipped,customer not in master,unknown_customer
10,1009,00073,NULL,8.00,1,shipped,NULL,missing_order_date
11,1010,00088,2024-03-04,9.99,NULL,shipped,NULL,missing_quantity
12,1011,00017,2024-03-05,10.00,two,shipped,NULL,unparseable_quantity
15,1013,00073,2024-03-05,100.00,1,shipped,NULL,conflicting_duplicate_order_id
16,1013,00073,2024-03-05,110.00,1,shipped,amended?,conflicting_duplicate_order_id
17,1014,17,2024-03-06,5.00,1,cancelled,NULL,unknown_customer
18,1015,00105,2024-03-06,3.00,1,lost_in_space,NULL,invalid_status


## 3. Join orders to customers
A `LEFT JOIN` keeps every valid order, even one whose customer has no country. `customers_clean` has one row per `customer_id` (step 1.5), so the join can't fan out. The tests check both.

In [20]:
step("orders_enriched", """
SELECT o.order_id, o.customer_id, c.country, o.order_date, o.amount, o.quantity, o.status, o.notes, o.source_row
FROM orders_clean o
LEFT JOIN customers_clean c ON c.customer_id = o.customer_id
ORDER BY o.order_id
""", kind="TABLE")

order_idVARCHAR,customer_idVARCHAR,countryVARCHAR,order_dateDATE,"amountDECIMAL(12,2)",quantityINTEGER,statusVARCHAR,notesVARCHAR,source_rowBIGINT
1001,00017,GB,2024-03-01,12.50,1,shipped,NULL,2
1002,00042,GB,2024-03-01,1200.00,2,shipped,"gift wrap, urgent",3
1003,00073,ES,2024-03-02,30.00,3,delivered,NULL,4
1004,00088,NA,2024-03-02,45.00,1,delivered,NULL,5
1008,00042,GB,2024-03-04,-5.00,1,refunded,NULL,9
1012,00042,GB,2024-03-05,7.25,1,shipped,NULL,13
1017,00105,NULL,2024-03-06,4.00,1,shipped,NULL,20
1018,00073,ES,2024-03-06,50.00,1,cancelled,NULL,21


## 4. Group, aggregate, sort
Revenue per day and country, without cancelled orders. Refunds count as negative amounts. In SQL a NULL `country` is its own group, so unlike a pandas `groupby` without `dropna=False`, that revenue doesn't vanish.

In [21]:
step("daily_revenue", """
SELECT order_date, country,
       sum(amount)              AS revenue,
       count(DISTINCT order_id) AS orders
FROM orders_enriched
WHERE status <> 'cancelled'
GROUP BY order_date, country
ORDER BY order_date, country NULLS LAST
""", kind="TABLE")

order_dateDATE,countryVARCHAR,"revenueDECIMAL(38,2)",ordersBIGINT
2024-03-01,GB,1212.50,2
2024-03-02,ES,30.00,1
2024-03-02,NA,45.00,1
2024-03-04,GB,-5.00,1
2024-03-05,GB,7.25,1
2024-03-06,NULL,4.00,1


## 5. Run end to end
`run()` replays every step on a fresh connection and checks that the rows reconcile **before** writing anything. Then it writes each output with `COPY ... TO`. The DuckDB docs say `COPY` overwrites an existing file and writes through a temp file first, so a rerun gives the same files and a failed write never leaves half a file.

In [22]:
OUTPUTS = ["customers_clean", "customers_quarantine", "orders_enriched", "orders_quarantine", "daily_revenue"]


def run(out_dir: Path = OUT_DIR) -> dict:
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    with duckdb.connect(config=CONFIG) as db:
        for sql in STEPS.values():
            db.execute(sql)

        def n(table: str) -> int:
            return db.execute(f"SELECT count(*) FROM {table}").fetchall()[0][0]

        # Reconciliation: every input row is accounted for, or we refuse to publish.
        exact_dups = n("orders_typed") - n("orders_dedup")
        accounted = n("orders_clean") + n("orders_quarantine") + exact_dups
        if accounted != n("orders_raw"):
            raise RuntimeError(f"row reconciliation failed: {n('orders_raw')} in, {accounted} out")

        for table in OUTPUTS:
            db.execute(f"COPY {table} TO '{out_dir / table}.csv' (HEADER)")

        reasons = db.sql("""
            SELECT reason, count(*) FROM (SELECT unnest(string_split(reject_reason, ';')) AS reason FROM orders_quarantine)
            GROUP BY reason ORDER BY reason
        """).fetchall()
        report = {
            "customers": {"rows_in": n("customers_raw"), "rows_out": n("customers_clean"),
                          "quarantined": n("customers_quarantine")},
            "orders": {
                "rows_in": n("orders_raw"),
                "valid": n("orders_clean"),
                "quarantined": n("orders_quarantine"),
                "exact_duplicates_dropped": exact_dups,
                "reject_reasons": dict(reasons),
            },
            "revenue_total": float(db.execute("SELECT sum(revenue) FROM daily_revenue").fetchall()[0][0]),
        }

    (out_dir / "run_report.json").write_text(json.dumps(report, indent=2, sort_keys=True) + "\n")
    return report

In [23]:
report = run()
report

{'customers': {'rows_in': 8, 'rows_out': 6, 'quarantined': 0},
 'orders': {'rows_in': 20,
  'valid': 8,
  'quarantined': 11,
  'exact_duplicates_dropped': 1,
  'reject_reasons': {'conflicting_duplicate_order_id': 2,
   'invalid_status': 1,
   'missing_order_date': 1,
   'missing_quantity': 1,
   'unknown_customer': 2,
   'unparseable_amount': 1,
   'unparseable_order_date': 2,
   'unparseable_quantity': 1}},
 'revenue_total': 1293.75}

## 6. Tests
The same checks as the pandas notebook, written the SQL way: each query returns the **offending rows**, so it passes when it returns nothing. A failure prints the bad rows straight away.

In [24]:
import filecmp
import tempfile
import unittest

EXPECT_NO_ROWS = {
    # --- 1. load ---------------------------------------------------------------
    "every raw column is text": """
        SELECT table_name, column_name, data_type FROM information_schema.columns
        WHERE table_name IN ('customers_raw', 'orders_raw') AND column_name <> 'source_row' AND data_type <> 'VARCHAR'""",
    "leading zeros preserved": "SELECT 'missing 00042' WHERE NOT EXISTS (SELECT 1 FROM customers_raw WHERE customer_id = '00042')",
    "literal NA is not missing": "SELECT 'missing NA' WHERE NOT EXISTS (SELECT 1 FROM customers_raw WHERE trim(country) = 'NA')",
    # --- 2. customers -------------------------------------------------------------
    "customer_id unique": "SELECT customer_id FROM customers_clean GROUP BY customer_id HAVING count(*) > 1",
    "latest version wins": "SELECT email FROM customers_clean WHERE customer_id = '00042' AND email <> 'robert.jones@example.com'",
    "strings tidied": """
        SELECT * FROM customers_clean
        WHERE email <> lower(email) OR country <> upper(trim(country)) OR name <> trim(name)""",
    "namibia survives": "SELECT country FROM customers_clean WHERE customer_id = '00088' AND country IS DISTINCT FROM 'NA'",
    "dates are dates": """
        SELECT column_name, data_type FROM information_schema.columns
        WHERE table_name = 'customers_clean' AND column_name = 'signup_date' AND data_type <> 'DATE'""",
    # --- 3. orders -------------------------------------------------------------------
    "every input row accounted for": """
        SELECT 'mismatch' WHERE (SELECT count(*) FROM orders_raw) <>
              (SELECT count(*) FROM orders_clean) + (SELECT count(*) FROM orders_quarantine)
            + (SELECT count(*) FROM orders_typed) - (SELECT count(*) FROM orders_dedup)""",
    "no order in both outputs": "SELECT order_id FROM orders_clean INTERSECT SELECT order_id FROM orders_quarantine",
    "column types": """
        SELECT column_name, data_type FROM information_schema.columns
        WHERE table_name = 'orders_clean'
          AND (column_name, data_type) NOT IN (('order_id', 'VARCHAR'), ('customer_id', 'VARCHAR'), ('order_date', 'DATE'),
              ('amount', 'DECIMAL(12,2)'), ('quantity', 'INTEGER'), ('status', 'VARCHAR'), ('notes', 'VARCHAR'),
              ('source_row', 'BIGINT'))""",
    "required fields not null": """
        SELECT * FROM orders_clean
        WHERE order_id IS NULL OR customer_id IS NULL OR order_date IS NULL OR amount IS NULL OR quantity IS NULL OR status IS NULL""",
    "order_id unique": "SELECT order_id FROM orders_clean GROUP BY order_id HAVING count(*) > 1",
    "status in allowed set": "SELECT status FROM orders_clean WHERE status NOT IN ('shipped', 'delivered', 'refunded', 'cancelled')",
    "negative amount only for refunds": "SELECT * FROM orders_clean WHERE amount < 0 AND status <> 'refunded'",
    "currency text parsed": "SELECT amount FROM orders_clean WHERE order_id = '1002' AND amount <> 1200.00",
    "every quarantined row has a reason": "SELECT * FROM orders_quarantine WHERE reject_reason IS NULL OR reject_reason = ''",
    "specific rejections": """
        SELECT e.* FROM (VALUES
            ('1005', 'unparseable_order_date'),  -- 30 February
            ('1006', 'unparseable_amount'),      -- "N/A"
            ('1007', 'unknown_customer'),
            ('1009', 'missing_order_date'),
            ('1011', 'unparseable_quantity'),    -- "two"
            ('1013', 'conflicting_duplicate_order_id'),
            ('1015', 'invalid_status'),
            ('1016', 'unparseable_order_date')   -- 06/03/2024 is ambiguous
        ) e(order_id, reason)
        WHERE NOT EXISTS (SELECT 1 FROM orders_quarantine q WHERE q.order_id = e.order_id AND contains(q.reject_reason, e.reason))""",
    "conflicting duplicates both quarantined": "SELECT 'not 2' WHERE (SELECT count(*) FROM orders_quarantine WHERE order_id = '1013') <> 2",
    "every valid order has a known customer": """
        SELECT o.order_id FROM orders_clean o
        WHERE NOT EXISTS (SELECT 1 FROM customers_clean c WHERE c.customer_id = o.customer_id)""",
    "join did not fan out": "SELECT 'fan-out' WHERE (SELECT count(*) FROM orders_enriched) <> (SELECT count(*) FROM orders_clean)",
    # --- 4. revenue -------------------------------------------------------------------
    "revenue reconciles with valid orders": """
        SELECT 'mismatch' WHERE (SELECT sum(revenue) FROM daily_revenue) <>
                                (SELECT sum(amount) FROM orders_enriched WHERE status <> 'cancelled')""",
    "unknown country group kept": "SELECT 'no NULL country' WHERE NOT EXISTS (SELECT 1 FROM daily_revenue WHERE country IS NULL)",
    "one row per day and country": "SELECT order_date, country FROM daily_revenue GROUP BY ALL HAVING count(*) > 1",
}


class DataTests(unittest.TestCase):
    def test_row_count_matches_file(self):
        for table in ["customers", "orders"]:
            with self.subTest(table):
                lines = (RAW / f"{table}.csv").read_text(encoding="utf-8").splitlines()
                rows = con.execute(f"SELECT count(*) FROM {table}_raw").fetchall()[0][0]
                self.assertEqual(rows, len(lines) - 1)


# One test per query, so each one is counted and named in the report.
for name, sql in EXPECT_NO_ROWS.items():
    def test(self, sql=sql):
        self.assertEqual(con.sql(sql).fetchall(), [], "offending rows")
    setattr(DataTests, "test_" + name.replace(" ", "_"), test)


class RunTests(unittest.TestCase):
    def test_idempotent_rerun(self):
        with tempfile.TemporaryDirectory() as a, tempfile.TemporaryDirectory() as b:
            run(Path(a))
            run(Path(a))  # second run over the same output dir
            run(Path(b))
            files = sorted(p.name for p in Path(a).iterdir())
            _, mismatch, errors = filecmp.cmpfiles(a, b, files, shallow=False)
            self.assertEqual((mismatch, errors), ([], []))
            self.assertEqual(sorted(p.name for p in Path(b).iterdir()), files)  # no stray temp files

    def test_report_reconciles(self):
        with tempfile.TemporaryDirectory() as d:
            report = run(Path(d))
            on_disk = json.loads((Path(d) / "run_report.json").read_text())
        o = report["orders"]
        self.assertEqual(o["rows_in"], o["valid"] + o["quarantined"] + o["exact_duplicates_dropped"])
        self.assertEqual(report, on_disk)

In [25]:
result = unittest.main(argv=["practice_duckdb"], exit=False, verbosity=2).result
failed, errors = len(result.failures), len(result.errors)
print(f"\n{result.testsRun - failed - errors} passed, {failed} failed, {errors} errors (of {result.testsRun})")
assert result.wasSuccessful()

test_column_types (__main__.DataTests.test_column_types) ... 

ok


test_conflicting_duplicates_both_quarantined (__main__.DataTests.test_conflicting_duplicates_both_quarantined) ... 

ok


test_currency_text_parsed (__main__.DataTests.test_currency_text_parsed) ... 

ok


... (46 outputs left out)

test_report_reconciles (__main__.RunTests.test_report_reconciles) ... 

ok


----------------------------------------------------------------------
Ran 27 tests in 0.196s

OK



27 passed, 0 failed, 0 errors (of 27)


## Notes

**Same decisions as the pandas notebook**: only ISO dates are accepted, and ambiguous ones are quarantined, not guessed. The latest customer version wins. Exact duplicates are dropped and counted, conflicting ones are both quarantined. A negative amount is valid only for a refund. Cancelled orders stay in the clean data but are excluded from revenue.

**Where DuckDB differs from pandas**
- Literal `NA` survives by default, and a NULL group key is kept by default.
- Casts are explicit: `try_strptime` and `TRY_CAST` give NULL instead of raising. But `TRY_CAST('1.5' AS INTEGER)` *rounds*, so quantity is checked first.
- Three-valued logic: `NOT IN` against a NULL silently matches nothing, so the rules use `IS NULL` and `NOT EXISTS`.
- Money is `DECIMAL(12, 2)`, so the revenue totals are exact.

**The four discussion questions**
- *Safe to run twice*: every step is `CREATE OR REPLACE`, every run starts from the raw files, and `COPY` overwrites through a temp file. The idempotency test proves the files come out byte-identical.
- *Rows you can't trust*: they go to `<table>_quarantine` with the raw text, `source_row` and a `reject_reason`. `run_report.json` counts every reason.
- *Knowing the output is right*: the reconciliation check (in = valid + quarantined + duplicates) stops the run before writing, the revenue total reconciles with the valid orders, and the tests check each trap.
- *Cost as data grows*: DuckDB streams the CSV, uses all cores and spills to disk above `memory_limit`. Next would be Parquet partitioned by date, loading only new `order_date` partitions, and `read_csv(..., store_rejects = true)` to quarantine malformed lines instead of failing on them.